# 03 : Comparadores estacionales y LightGBM

Se evalúan cero, estacional semanal, promedio de cuatro semanas y LightGBM sobre los mismos cortes y claves. La métrica principal es RMSLE agrupado bajo `asof_v1`.

## Ejecución de la entrega

Ejecuta todas las celdas en orden en una sesión limpia. El modo predeterminado
`full` utiliza datos reales; el notebook 04 requiere GPU. Para comprobar solo
esquema y ejecución, define `os.environ["RETAIL_FORECAST_MODE"] = "smoke"` antes
de la preparación. Los resultados sintéticos no miden precisión.

Mantén los siete CSV en `data/raw` del espacio de trabajo o usa el ZIP de Kaggle.
No escribas credenciales en celdas ni salidas. Conserva el notebook ejecutado y
el directorio de resultados antes de desconectar Colab.

In [ ]:
# Pinned source setup; Colab preserves its compatible resident libraries.
import importlib.util
import os
import re
import runpy
import subprocess
import sys
import tempfile
from getpass import getpass
from pathlib import Path

# The published academic URL can be supplied through the environment.
REPO_URL = os.environ.get("RETAIL_FORECAST_REPO_URL", "").strip() or globals().get("REPO_URL", "")
if not REPO_URL:
    if (Path.cwd() / "pyproject.toml").is_file():
        REPO_URL = str(Path.cwd())
    else:
        REPO_URL = "https://github.com/camiloandcu/Retail-Demand-Forecasting.git"
if not REPO_URL:
    raise ValueError("Indica la URL del repositorio académico publicado.")
CODE_REVISION = "8353a8d9937e4d8061b026d6885296fcd6b09726"
MODELS = True
if sys.version_info[:2] not in {(3, 10), (3, 11), (3, 12), (3, 13)}:
    raise RuntimeError("Use Python 3.10–3.13 before installing or acquiring data.")
if not re.fullmatch(r"[0-9a-f]{40}", CODE_REVISION):
    raise RuntimeError("CODE_REVISION must be a reviewed full commit SHA.")
IN_COLAB = (
    importlib.util.find_spec("google") is not None
    and importlib.util.find_spec("google.colab") is not None
)
REPO_ROOT = Path(
    os.environ.get(
        "RETAIL_FORECAST_CHECKOUT",
        str(Path("/content" if IN_COLAB else tempfile.gettempdir()) / f"retail-academic-{CODE_REVISION}"),
    )
).absolute()
if REPO_ROOT.is_symlink() or any(p.is_symlink() for p in REPO_ROOT.parents):
    raise RuntimeError("Checkout path must not contain symlinks; use a new directory.")


def checkout_git(*arguments):
    return subprocess.run(
        ["git", "-C", str(REPO_ROOT), *arguments],
        check=True,
        capture_output=True,
        text=True,
    ).stdout.strip()


if not REPO_ROOT.exists():
    subprocess.run(["git", "clone", "--no-checkout", REPO_URL, str(REPO_ROOT)], check=True)
    # A squash merge can leave the reviewed SHA reachable only by its PR ref.
    subprocess.run(
        ["git", "-C", str(REPO_ROOT), "fetch", "--no-tags", "origin", CODE_REVISION],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(REPO_ROOT), "checkout", "--detach", CODE_REVISION],
        check=True,
    )
# Validate before executing even the shared helper from this checkout.
try:
    valid = (
        checkout_git("rev-parse", "--show-toplevel") == str(REPO_ROOT)
        and checkout_git("remote", "get-url", "origin") == REPO_URL
        and checkout_git("rev-parse", "HEAD") == CODE_REVISION
        and not checkout_git("status", "--porcelain", "--untracked-files=all")
    )
except subprocess.CalledProcessError:
    valid = False
if not valid:
    raise RuntimeError(
        "Dirty, incomplete or mismatched checkout. Preserve it; use a new directory."
    )
helper = runpy.run_path(str(REPO_ROOT / "src/retail_forecast/notebook_runtime.py"))
runtime_evidence = helper["bootstrap"](
    REPO_ROOT, CODE_REVISION, REPO_URL, models=MODELS, colab=IN_COLAB
)
print(
    "Notebook revision:",
    os.getenv("RETAIL_FORECAST_NOTEBOOK_REVISION", "unrecorded; retain immutable notebook URL"),
)
WORKSPACE_ROOT = globals().get(
    "WORKSPACE_ROOT",
    Path("/content") if IN_COLAB else (
        Path.cwd() if (Path.cwd() / "pyproject.toml").is_file() else REPO_ROOT
    ),
)
os.chdir(REPO_ROOT)

In [ ]:
from dataclasses import replace

import pandas as pd
from IPython.display import Image, display

from retail_forecast.acquisition import acquire_competition_data
from retail_forecast.artifacts import (
    capture_inputs,
    finish_run,
    read_completed_run,
    run_result,
    write_run_artifact,
)
from retail_forecast.config import load_config
from retail_forecast.data import (
    required_files_exist,
    validate_dataset,
    write_synthetic_dataset,
)
from retail_forecast.eda import validate_kaggle_hashes
from retail_forecast.experiments import MODEL_IDS, run_baseline_experiments
from retail_forecast.notebook_runs import notebook_config_path, prepare_notebook_run
from retail_forecast.reproducibility import set_global_seed

MODE = os.getenv("RETAIL_FORECAST_MODE", "full").strip().lower()
# The information regime is explicit; results from different policies never pool.
POLICY = os.getenv("RETAIL_FORECAST_POLICY", "asof_v1").strip().lower()
config = load_config(notebook_config_path(MODE, POLICY))
config = replace(
    config,
    paths=replace(
        config.paths,
        raw_data_dir=WORKSPACE_ROOT / config.paths.raw_data_dir.relative_to(REPO_ROOT),
        artifacts_dir=WORKSPACE_ROOT / config.paths.artifacts_dir.relative_to(REPO_ROOT),
    ),
)
run_context, config = prepare_notebook_run(config, kind="baseline")
print("Run ID:", run_context.run_id, "| root:", run_context.root)
set_global_seed(config.runtime.seed)
print(
    f"mode={MODE} | policy={config.data.availability_policy} "
    f"| configurations={len(MODEL_IDS)} | tree={config.baseline.tree_model}"
)

## Datos

Se usan los CSV validados del espacio de trabajo; si faltan, se sigue el procedimiento de adquisición de la guía.

In [ ]:
if config.data.source == "synthetic":
    if not required_files_exist(config.paths.raw_data_dir):
        write_synthetic_dataset(config)
elif not required_files_exist(config.paths.raw_data_dir):
    acquire_competition_data(
        config.paths.raw_data_dir,
        (WORKSPACE_ROOT / "data", WORKSPACE_ROOT),
        token_reader=getpass,
    )
capture_inputs(run_context, input_dir=config.paths.raw_data_dir)
if MODE == "full":
    validate_kaggle_hashes(config.paths.raw_data_dir)
dataset_summary = validate_dataset(config)
display(dataset_summary.to_dict())

## Receta fijada

LightGBM es el único modelo de árboles. Aprende `log1p(ventas)` y usa categorías e imputación ajustadas dentro de cada entrenamiento. Sus entradas no incluyen petróleo bajo `asof_v1`. Las referencias semanales prolongan la historia sin leer etiquetas de validación.

In [ ]:
run = run_baseline_experiments(
    config,
    run=run_context,
    tracking=True,
)

## Resultados y costes

Se muestran errores globales, por corte y por día, además de tiempos y artefactos. Un umbral del 2 % que aparece en metadatos históricos no es un criterio de aprobación de esta entrega. Las métricas sintéticas no miden precisión sobre Favorita.

In [ ]:
display(run["cost_quality"])
display(
    run["results"]
    .loc[run["results"]["scope"].isin(["global", "fold"])]
    .pivot_table(index=["scope", "segment"], columns="model_id", values="rmsle")
)
display(
    run["results"]
    .loc[run["results"]["scope"].eq("horizon")]
    .assign(segment=lambda frame: frame["segment"].astype(int))
    .pivot(index="segment", columns="model_id", values="rmsle")
)
display(Image(filename=str(run_context.root / "figures/baselines/01_global_rmsle.png")))
display(Image(filename=str(run_context.root / "figures/baselines/02_horizon_rmsle.png")))

In [ ]:
import matplotlib.pyplot as plt

horizon_without_zero = (
    run["results"]
    .loc[run["results"]["scope"].eq("horizon") & run["results"]["model_id"].ne("zero")]
    .assign(segment=lambda frame: frame["segment"].astype(int))
    .sort_values(["model_id", "segment"])
)

labels = {
    "seasonal_naive_7d": "Seasonal naive (7d)",
    "seasonal_average_4w": "Seasonal average (4w)",
    "lightgbm_global": "LightGBM",
}
figure, axis = plt.subplots(figsize=(9, 4.5))
for model_id, model_frame in horizon_without_zero.groupby("model_id", sort=False):
    axis.plot(
        model_frame["segment"],
        model_frame["rmsle"],
        marker="o",
        label=labels[model_id],
    )
axis.set(
    xlabel="Forecast horizon",
    ylabel="OOF RMSLE",
    title="Error by forecast horizon (excluding the zero check)",
)
axis.set_xticks(sorted(horizon_without_zero["segment"].unique()))
axis.grid(axis="y", alpha=0.25)
axis.legend()
figure.tight_layout()

figure_relative = "figures/baselines/02_horizon_rmsle_without_zero.png"
write_run_artifact(
    run_context,
    figure_relative,
    lambda path: figure.savefig(path, format="png", dpi=140, bbox_inches="tight"),
    kind="figure",
)
figure_path = run_context.root / figure_relative
plt.show()
print(f"Saved: {figure_path}")

## Lectura de la ejecución real registrada

En Colab, LightGBM obtuvo RMSLE 0,398680 y el promedio de cuatro semanas 0,468430. El control cero obtuvo 4,419661; se aparta de los gráficos detallados para que no oculte la diferencia entre modelos útiles.

LightGBM tuvo menor error que las referencias estacionales en los 16 horizontes. Su RMSLE pasó de 0,378975 el primer día a 0,417610 el día 16, con fluctuaciones intermedias. El error por horizonte también depende de las fechas que representa. Si se ejecuta otra receta o el modo sintético, se deben leer sus salidas por separado.

## Referencias para la GRU

El promedio estacional ofrece una comparación simple y LightGBM una referencia aprendida más fuerte. La GRU se comparará sobre las mismas claves, cortes y política de información; su utilidad no se da por supuesta.

In [ ]:
display(run["manifest"])
for name, path in run["paths"].items():
    print(f"{name}: {path}")

## Errores por segmento

Se revisan periodos, promociones, tiendas y familias. Los segmentos se calculan desde las predicciones ya obtenidas; no constituyen nuevos experimentos independientes.

In [ ]:
tree_segments = run["results"].loc[run["results"]["model_id"].eq("lightgbm_global")]
for scope in ("promotion", "temporal_regime"):
    print(f"LightGBM by {scope}")
    display(tree_segments.loc[tree_segments["scope"].eq(scope), ["segment", "rows", "rmsle"]])
for scope in ("store", "family"):
    print(f"Five highest-error {scope} segments")
    display(
        tree_segments.loc[tree_segments["scope"].eq(scope), ["segment", "rows", "rmsle"]]
        .sort_values("rmsle", ascending=False)
        .head(5)
    )
print(f"Complete segment results: {run['paths']['results']}")

### Dónde conviene mirar

En la ejecución registrada, SCHOOL AND OFFICE SUPPLIES tiene RMSLE 0,658960 con LightGBM, frente a 0,661438 en GROCERY II. En 04 se pueden contrastar con la GRU sobre esas mismas familias. Las diferencias entre filas promovidas o festivas no identifican causas: cambian también tiendas, familias, fechas y ventas.

In [ ]:
checks = {
    "four configurations only": len(MODEL_IDS) == 4,
    "one tree family only": config.baseline.tree_model == "lightgbm",
    "official result artifact": run["paths"]["results"].is_file(),
    "OOF artifact": run["paths"]["oof"].is_file(),
    "tracking status recorded": run["manifest"]["tracking"]["status"]
    in {"enabled", "disabled", "failed"},
}
gate = pd.DataFrame(
    [{"check": name, "status": "PASS" if passed else "FAIL"} for name, passed in checks.items()]
)
display(gate)
if not all(checks.values()):
    raise RuntimeError("Baseline gate failed.")
print(f"Baseline execution checks: {MODE.upper()} PASS")
finish_run(run_context)
verified_run = read_completed_run(
    run_context.root,
    expected_mode=MODE,
    expected_source=config.data.source,
)
print("Verified completed run:", run_result(run_context))